# Polity assignment
How individuals are assigned to Cliopatria polities, by assignment method.

In [1]:
import tempfile

import duckdb
import polars as pl

DB_PATH = '../data/cultura/humans_clean_enriched_v3.duckdb'

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(60)

polars.config.Config

## Load data
### Matches per assignment method

In [2]:
con = duckdb.connect(DB_PATH, read_only=True)
con.sql(f"SET memory_limit='4GB'; SET threads=6; SET temp_directory='{tempfile.mkdtemp()}';")

method_counts = con.sql("""
    SELECT   match.assignation_method          AS assignation_method,
             COUNT(*)                          AS n_matches,
             COUNT(DISTINCT entity.qid)        AS n_individuals
    FROM     individual_enriched, unnest(polity) AS unnested(match)
    GROUP BY match.assignation_method
""").pl()
method_counts

assignation_method,n_matches,n_individuals
str,i64,i64
"""url_of_deathplace""",319,304
"""polygon_of_birthplace""",1009884,709684
"""url_of_country_of_citizenship""",3226081,3181006
"""url_of_birthplace""",432,427
"""polygon_of_country_of_citizenship""",577427,444084
"""polygon_of_deathplace""",1364690,814694


### Denominators

In [3]:
denominators = con.sql("""
    SELECT COUNT(*)                                                    AS n_individuals,
           COUNT(peak_productivity.start_year)                         AS n_dated,
           COUNT(*) FILTER (WHERE len(polity) > 0)                     AS n_matched
    FROM   individual_enriched
""").pl().row(0, named=True)
con.close()

assert denominators['n_matched'] == method_counts['n_individuals'].sum(), 'each individual is matched by a single method'
for key, value in denominators.items():
    print(f'{key:14s}: {value:,}')

n_individuals : 13,002,897
n_dated       : 9,823,124
n_matched     : 5,150,199


## Table: individual-to-polity assignment by method
Methods are listed in cascade order: an individual is assigned by the first method that matches.

In [4]:
CASCADE_ORDER = [
    ('polygon_of_deathplace',             'Polygon merging', 'Place of death'),
    ('polygon_of_birthplace',             'Polygon merging', 'Place of birth'),
    ('polygon_of_country_of_citizenship', 'Polygon merging', 'Country of citizenship'),
    ('url_of_country_of_citizenship',     'URL matching',    'Country of citizenship'),
    ('url_of_deathplace',                 'URL matching',    'Place of death'),
    ('url_of_birthplace',                 'URL matching',    'Place of birth'),
]

assignment_table = (
    pl.DataFrame(CASCADE_ORDER, schema=['assignation_method', 'method', 'origin'], orient='row')
    .with_row_index('step', offset=1)
    .join(method_counts, on='assignation_method', how='left')
    .with_columns(pl.col('n_matches', 'n_individuals').fill_null(0))
    .with_columns(
        (pl.col('n_individuals') / denominators['n_matched'] * 100).alias('share_of_matched_pct'),
        (pl.col('n_individuals') / denominators['n_dated'] * 100).alias('share_of_dated_pct'),
        (pl.col('n_individuals') / denominators['n_individuals'] * 100).alias('share_of_total_pct'),
    )
    .with_columns(pl.col('^.*_pct$').round(2))
    .select('step', 'method', 'origin', 'n_matches', 'n_individuals',
            'share_of_matched_pct', 'share_of_dated_pct', 'share_of_total_pct')
)

assert assignment_table['n_individuals'].sum() == denominators['n_matched']
assignment_table

step,method,origin,n_matches,n_individuals,share_of_matched_pct,share_of_dated_pct,share_of_total_pct
u32,str,str,i64,i64,f64,f64,f64
1,"""Polygon merging""","""Place of death""",1364690,814694,15.82,8.29,6.27
2,"""Polygon merging""","""Place of birth""",1009884,709684,13.78,7.22,5.46
3,"""Polygon merging""","""Country of citizenship""",577427,444084,8.62,4.52,3.42
4,"""URL matching""","""Country of citizenship""",3226081,3181006,61.76,32.38,24.46
5,"""URL matching""","""Place of death""",319,304,0.01,0.0,0.0
6,"""URL matching""","""Place of birth""",432,427,0.01,0.0,0.0
